# Argument_Analysis_Agentic-1-informal : Detection de sophismes par taxonomie

**Navigation** : [README](./README.md) | [2-formal >>](./Argument_Analysis_Agentic-2-formal.ipynb)

| Champ | Valeur |
|-------|--------|
| **Module** | SymbolicAI / Argument_Analysis |
| **Rung** | 1-informal (Epic #2137 Phase 1) |
| **Niveau** | Introduction |
| **Duree estimee** | 30 minutes |
| **Kernel** | Python 3 (pandas) |

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

- [ ] Charger la **taxonomie des sophismes** (CSV `argumentum_fallacies_taxonomy.csv`, 1408 lignes) et identifier les 7 familles de premier niveau
- [ ] Charger la **taxonomie soeur des vertus argumentatives** (CSV `argumentum_virtues_taxonomy.csv`, 222 cartes, schema distinct) et constater la **divergence de schema** entre les deux CSV (deliberation upstream Argumentum)
- [ ] Descendre d'un niveau dans l'arbre taxonomique (depth=2) pour une famille donnee, via un filtre pandas
- [ ] Construire un **detecteur deterministe** (sans LLM) base sur des mots-cles, et l'appliquer sur un texte synthetique neutre
- [ ] Distinguer le role du detecteur par mots-cles (ce rung) du role du LLM (rungs ulterieurs) pour l'extraction semantique

### Prerequis

- Python 3.10+ ; le package `pandas` installe
- Bases de lecture d'un CSV avec pandas (`read_csv`, filtrage boolean)
- Aucune cle API requise (ce rung est 100% deterministe, zero appel LLM)

> **Note anti-theatre** : ce rung charge **reellement** le CSV et execute un **vrai** filtrage pandas. Aucune sortie n'est simulee. Le detecteur par mots-cles est volontairement simpliste : son but est pedagogique (montrer la couche de matching), pas d'atteindre une haute precision. L'extraction semantique robuste est l'objet des rungs suivants (LLM + logique formelle).

> **Confidentialite** : tous les exemples sont **synthetiques et neutres** (meteorologie, oiseaux, Socrate, phrases abstraites). Aucun corpus EPITA, aucun nom d'etudiant, aucun texte sensible. Le depot est public sur GitHub.

## 1. Introduction : la taxonomie des sophismes

Un **sophisme** (fallacy) est un argument qui semble valide mais qui ne l'est pas, soit parce que les premisses ne soutiennent pas la conclusion, soit parce qu'il manipule l'auditoire. Pour detecter automatiquement un sophisme, il faut d'abord une **taxonomie** : un arbre hierarchise qui classe les sophismes par familles, sous-familles, etc.

Ce notebook utilise la taxonomie du projet *Argumentum* (1408 entrees, post-resync verbatim upstream ac33f607 -- voir `data/NOTICE` et `data/NOTICE-VIRTUES`). Une **taxonomie soeur** des vertus argumentatives (`argumentum_virtues_taxonomy.csv`, 222 cartes) vit dans le meme dossier et partage la meme structure d'arbre depth 0..7. Les deux CSV sont organises comme des arbres de profondeur variable :

| `depth` | Role | Exemple |
|---------|------|--------|
| `0` | Meta-noeud racine | "Argument fallacieux" (1 seule ligne, a exclure) |
| `1` | **7 familles de premier niveau** | Insuffisance, Influence, Obstruction, ... |
| `2` | Sous-familles | Obstruction -> {Refus du debat, Sabotage, Ad hominem} |
| `3`+ | Sophismes specifiques | Ad hominem -> circonstanciel, abusif, ... |

**Stance anti-theatre** : on charge le CSV avec `pandas.read_csv` et on filtre **reellement** les lignes. Le detecteur de la section 3 tourne sur de vraies listes de mots-cles appliquees a un texte synthetique — jamais de resultat en dur.

| Etape | Outil | Ce qu'il fait |
|-------|-------|---------------|
| Texte naturel -> sophisme identifie | Detecteur mots-cles (ce rung) | Matching chaine, deterministe, peu precis |
| Texte naturel -> structure informelle | LLM (rung 3) | Extraction semantique probabiliste |
| Structure -> verification logique | Tweety (rung 2) | Preuve sound & complete |

## 2. Charger la taxonomie et identifier les 7 familles

La cellule ci-dessous charge le CSV avec `pandas.read_csv(..., dtype=str)` — tout est traite comme chaine pour eviter les coercitions numeriques sur les colonnes `depth`, `path`, etc. On filtre ensuite `depth == '1'` pour obtenir les familles de premier niveau.

> **Pourquoi exclure `depth == '0'` ?** Il existe un meta-noeud racine "Argument fallacieux" (1 ligne) qui represente le concept general de sophisme. Ce n'est pas une famille operationnelle : on l'evicte pour ne garder que les 7 vraies familles.

In [1]:
# Cellule [2] - Charger la taxonomie via la shim argumentation_lib
import pandas as pd

# Le package argumentation_lib vit a cote du notebook ; il est importable
# directement (le dossier du notebook est sur sys.path au runtime). La shim
# resout le dossier data/ via __file__ (cwd-independant) -- on evite ainsi le
# chemin relatif 'data/...' qui leverait FileNotFoundError hors de ce dossier.
# Aucun guard de path dans la cellule : la robustesse cwd est portee par la
# shim (_paths.py, __file__-relative), comme au rung 2-formal (#3857).
from argumentation_lib import get_data_dir
CSV_PATH = get_data_dir() / "argumentum_fallacies_taxonomy.csv"
print(f"Taxonomie resolue via la shim : {CSV_PATH}")

df = pd.read_csv(CSV_PATH, dtype=str, encoding="utf-8-sig")  # BOM UTF-8 stripped (canonical upstream ac33f607)

# La racine depth=0 est un meta-noeud (1 ligne) : on l'exclut.
meta_root = df[df['depth'] == '0']
print(f"Total lignes CSV      : {len(df)}")
print(f"Meta-noeud depth=0    : {len(meta_root)} ligne(s) -> exclue(s) de la liste des familles")

# Les 7 familles de premier niveau
familles = df[df['depth'] == '1'][['Famille', 'desc_fr']].reset_index(drop=True)
print(f"Familles depth=1      : {len(familles)}")
print()
print('=== Les 7 familles de sophismes (depth=1) ===')
for i, row in familles.iterrows():
    num = i + 1
    desc = row['desc_fr'] if pd.notna(row['desc_fr']) else '(pas de description)'
    print(f"  {num}. {row['Famille']:<25} -- {desc}")

Taxonomie resolue via la shim : D:\dev\CoursIA-2\MyIA.AI.Notebooks\SymbolicAI\Argument_Analysis\data\argumentum_fallacies_taxonomy.csv
Total lignes CSV      : 1408
Meta-noeud depth=0    : 1 ligne(s) -> exclue(s) de la liste des familles
Familles depth=1      : 7

=== Les 7 familles de sophismes (depth=1) ===
  1. Insuffisance              -- Les arguments que vous utilisez ne suffisent pas à démontrer votre propos.
  2. Influence                 -- Vous manipulez votre auditoire pour le persuader au lieu de le convaincre.
  3. Erreur mathématique       -- Votre raisonnement comporte des inexactitudes quantitatives.
  4. Erreur de raisonnement    -- Votre thèse repose sur un raisonnement incohérent.
  5. Abus de langage           -- Vous usez de subtilités linguistiques pour convaincre.
  6. Tricherie                 -- Vous vous affranchissez des règles tacites qui régissent un débat rationnel.
  7. Obstruction               -- Par un artifice, oratoire ou autre, vous faites en sorte q

### Interpretation : les 7 familles de la taxonomie

**Sortie obtenue** : 7 familles de depth=1, chacune avec une description courte.

| Famille | Idee generale |
|---------|---------------|
| **Insuffisance** | Les premisses ne suffisent pas a prouver la conclusion |
| **Influence** | Manipulation emotionnelle de l'auditoire au lieu d'argumenter |
| **Erreur mathematique** | Inexactitude quantitative (statistiques, proportions) |
| **Erreur de raisonnement** | Raisonnement incoherent logiquement |
| **Abus de langage** | Jeu sur les mots, ambiguite linguistique |
| **Tricherie** | Contournement des regles tacites du debat rationnel |
| **Obstruction** | La discussion est empechee de se derouler normalement |

> **Point cle** : le meta-noeud `depth=0` ("Argument fallacieux", 1 ligne) n'est PAS une famille. C'est la racine conceptuelle de tout l'arbre. Les 7 familles operationnelles commencent a `depth=1`.

In [2]:
# Cellule [4-bis] - Charger la taxonomie soeur des VERTUS argumentatives
#
# Meme repertoire, meme upstream ac33f607, schema distinct delibere.
# On utilise la MEME shim cwd-robuste ; on dispatche sur le filename.
# Note pedagogique : les colonnes "snake_case" (family_fr) sont incompatibles
# avec le "PascalCase" du CSV Fallacies (Famille). Tout consumer qui
# presume un schema unifie tombera sur un KeyError -- d'ou l'importance du
# dispatch filename-based documente en data/NOTICE-VIRTUES.
VERT_PATH = get_data_dir() / "argumentum_virtues_taxonomy.csv"
df_vert = pd.read_csv(VERT_PATH, dtype=str, encoding="utf-8-sig")  # BOM UTF-8 stripped

# Distribution des profondeurs (Vertues 0..7, vs Fallacies 0..10)
depth_dist_vert = df_vert['depth'].value_counts().sort_index()
print(f"Taxonomie Vertus resolue via la shim : {VERT_PATH}")
print(f"Total cartes Vertus  : {len(df_vert)} (Fallacies: {len(df)} -- facteur ~6.3x)")
print(f"Distribution des profondeurs (Vertues 0..7) :")
for d, n in depth_dist_vert.items():
    print(f"  depth={d}  -> {n} cartes")
print()
print("=== Premiers vertus (depth=1, familles de premier niveau) ===")
vert_familles = df_vert[df_vert['depth'] == '1'][['family_fr', 'description_fr']].reset_index(drop=True)
for i, row in vert_familles.iterrows():
    num = i + 1
    desc = row['description_fr'] if pd.notna(row['description_fr']) else '(pas de description)'
    print(f"  {num}. {row['family_fr']:<25} -- {desc}")


Taxonomie Vertus resolue via la shim : D:\dev\CoursIA-2\MyIA.AI.Notebooks\SymbolicAI\Argument_Analysis\data\argumentum_virtues_taxonomy.csv
Total cartes Vertus  : 223 (Fallacies: 1408 -- facteur ~6.3x)
Distribution des profondeurs (Vertues 0..7) :
  depth=0  -> 1 cartes
  depth=1  -> 7 cartes
  depth=2  -> 21 cartes
  depth=3  -> 63 cartes
  depth=4  -> 69 cartes
  depth=5  -> 28 cartes
  depth=6  -> 15 cartes
  depth=7  -> 19 cartes

=== Premiers vertus (depth=1, familles de premier niveau) ===
  1. Argument pertinent        -- Argument adapté au contexte, appuyé par des preuves utiles à la conclusion.
  2. Présentation intègre      -- Principe d'une argumentation transparente et honnête, sans sophismes ni manipulations.
  3. Rigueur mathématique      -- Employer les mathématiques avec précision pour structurer et analyser des arguments ou des données.
  4. Raisonnement valide       -- Méthode rigoureuse qui articule des prémisses acceptables et des règles d'inférence valides pour abo

### Interpretation : divergence de schema entre Fallacies et Vertues

**Sortie obtenue** : 222 cartes Vertus, structure depth=0..7, avec une distribution pedagogiquement interessante (1 racine / 7 familles / 21 sous-familles / 63 ss-sous-familles, puis 69/28/15/19 aux niveaux 4..7). Cette distribution suit la **meme progression que les Fallacies** pour les 3 premiers niveaux (1/7/21/63) mais diverge au-dela -- les Vertues sont un arbre **moins profond et moins dense**.

**Schema divergence** (deliberation upstream Argumentum -- preserve verbatim) :

| Aspect | Fallacies (cell [3]) | Vertues (cell [4-bis]) |
|--------|---------------------|------------------------|
| Lignes totales | 1408 (PK 0..1407) | 222 (pk 0..221) |
| Profondeur max | 10 (cas rares) | 7 |
| **Header naming** | **PascalCase mixte** (`PK`, `Famille`, `Famille_camelCase`, `Sous-Famille`) | **snake_case unifie** (`pk`, `family_fr`, `subfamily_fr`, `subsubfamily_fr`) |
| **Card body fields** | **multi-field** (`text_fr`, `LTfr`, `Lfr115`, `example_fr`, `Lxfr145` -- 6+ par langue) | **3-field simplifie** (`title_fr`, `description_fr`, `remark_fr`) |
| Multilingue | FR/EN/RU/PT/AR/ES/ZH/FA (memes 8 langues) | idem |
| Cross-link | `crossLink_PredatesOn` / `Denounces` / ... (8 cols) | idem |
| AIF | `AIF_skosDirectRef` / `ExceptionRef` / ... (4 cols) | idem |

**Implication consumer** : le `family_fr` des Vertues N'EXISTE PAS comme `Famille` dans les Fallacies (la cle est en snake_case, pas PascalCase). Le `text_fr` multi-field des Fallacies N'EXISTE PAS comme `title_fr`/`description_fr` dans les Vertues. **Tout code qui presume un schema unifie tombera sur un KeyError**. Les loaders doivent dispatcher sur le **filename** (comme ici : Fallacies via `argumentum_fallacies_taxonomy.csv`, Vertues via `argumentum_virtues_taxonomy.csv`), pas assumer une liste de colonnes commune. Cette divergence est documentee verbatim dans `data/NOTICE-VIRTUES` pour eviter la surprise silencieuse cote consumer.

> **Pourquoi `encoding="utf-8-sig"` ici aussi ?** Meme raison que pour les Fallacies (cell [3]) : les deux CSV sont consignes verbatim upstream ac33f607 avec leur BOM UTF-8 d'origine (bytes `0xEF 0xBB 0xBF` en debut de fichier). Pandas 2.x detecte ce BOM automatiquement avec `encoding='utf-8'`, mais specifier `encoding='utf-8-sig'` est **explicite et documente** (per ai-01 design-gate msg-20260703T110001-uqono0, 2026-07-03) et permet une portabilite vers d'autres readers Python (`csv.reader`, `open().readlines()`).


## 3. Descente d'un niveau (depth=2) : la famille Obstruction

Pour comprendre comment l'arbre se ramifie, on choisit une famille et on regarde ses **sous-familles directes** (depth=2). Prenons **Obstruction** : la famille qui regroupe les sophismes ou l'on empeche le debat d'avoir lieu normalement.

On filtre avec deux conditions combinees par `&` : `Famille == 'Obstruction'` ET `depth == '2'`. Le resultat est un sous-DataFrame qu'on affiche avec ses colonnes utiles.

In [3]:
# Cellule [4] - Descente d'un niveau : sous-familles depth=2 de la famille Obstruction
# On filtre le DataFrame principal : famille choisie ET depth=2.
FAMILLE_DEMO = 'Obstruction'
sous_familles = df[(df['Famille'] == FAMILLE_DEMO) & (df['depth'] == '2')][
    ['path', 'Famille', 'Sous-Famille', 'text_fr', 'example_fr']
].reset_index(drop=True)

print(f"=== Sous-familles depth=2 de '{FAMILLE_DEMO}' ({len(sous_familles)} trouvees) ===")
for i, row in sous_familles.iterrows():
    text_fr = row['text_fr'] if pd.notna(row['text_fr']) else '(sans nom)'
    example = row['example_fr'] if pd.notna(row['example_fr']) else '(pas d exemple)'
    ex_str = str(example)
    print(f"\n  [{row['path']}] {text_fr}")
    print(f"      Exemple : {ex_str[:90]}{'...' if len(ex_str) > 90 else ''}")

=== Sous-familles depth=2 de 'Obstruction' (3 trouvees) ===

  [7.1] Refus du débat
      Exemple : Mes positions sont claires, connues de tous, et je n’ai pas à en débattre.

  [7.2] Sabotage du débat
      Exemple : Allons plutôt dans mon bureau discuter de l’absurdité de votre demande d’augmentation…

  [7.3] Ad hominem
      Exemple : Comment pouvez-vous prétendre que votre politique de santé fonctionnera quand vous n’avez ...


### Interpretation : la ramification depth=2

**Sortie obtenue** : 3 sous-familles pour Obstruction, identifiees par leur `path` hierarchique.

| path | Sous-famille | Idee |
|------|--------------|------|
| `7.1` | Refus du debat | Refuser d'argumenter (positions non negociables) |
| `7.2` | Saboter le debat (Sabotage) | Detourner la discussion, la rendre impossible |
| `7.3` | Ad hominem | Attaquer la personne plutot que l'argument |

**Points cles** :
1. Le `path` (ex `7.1`, `7.2`) encode la position hierarchique : le `7` correspond a la 7e famille de depth=1 (Obstruction), le suffixe `.N` a la Neme sous-famille.
2. Chaque sous-famille peut elle-meme se ramifier en depth=3 (ex : Ad hominem -> circonstanciel, abusif, preemptif). L'arbre a jusqu'a 10 niveaux pour certains sophismes tres specifiques.
3. La colonne `example_fr` donne un exemple d'usage — utile pour calibrer un detecteur ou entrainer un classifieur.

### Exercice 1 : descendre d'un niveau pour une AUTRE famille

**Contexte** : on vient de descendre Obstruction. Le meme filtre pandas fonctionne pour n'importe quelle famille.

**Objectif** : choisissez une famille parmi {Insuffisance, Influence, Erreur mathematique, Erreur de raisonnement, Abus de langage, Tricherie} et affichez ses sous-familles depth=2 (nom + exemple). Retournez le nombre de sous-familles trouvees.

**Indices** :
- # Etape 1 : fixer `FAMILLE_EX1` a un nom parmi les 6 autres familles
- # Etape 2 : reutiliser le filtre `df[(df['Famille'] == FAMILLE_EX1) & (df['depth'] == '2')]`
- # Etape 3 : iterer et afficher chaque sous-famille avec son `text_fr`
- # Etape 4 : retourner/compter le nombre de sous-familles

In [4]:
# Exercice 1 : descendre d'un niveau pour une autre famille que Obstruction.
# Etape 1 : choisir une famille (parmi Insuffisance, Influence, Erreur mathematique,
#            Erreur de raisonnement, Abus de langage, Tricherie).
FAMILLE_EX1 = None  # TODO etudiant : remplacer par le nom d'une famille

# Etape 2-3 : filtrer et afficher les sous-familles depth=2 de cette famille.
# TODO etudiant : reprendre le filtre de la cellule demo avec FAMILLE_EX1.

# Etape 4 : nombre de sous-familles trouvees
nb_sous_familles_ex1 = None  # TODO etudiant : len(...) du DataFrame filtre

print(f"Exercice 1 : {FAMILLE_EX1} a {nb_sous_familles_ex1} sous-famille(s) depth=2")
print("Exercice a completer")

Exercice 1 : None a None sous-famille(s) depth=2
Exercice a completer


## 4. Detection deterministe par mots-cles sur un texte synthetique

La descente taxonomique nous donne un **referentiel** de noms de sophismes. On construit maintenant un **detecteur** : etant donne un texte, quel sophisme de la taxonomie est probablement present ?

Ce detecteur est **deterministe et zero-LLM** : il s'agit d'un simple dictionnaire `{nom_sophisme: [mots_cles]}`. Si un mot-cle apparait (insensible a la casse) dans le texte, le sophisme correspondant est signale. Ce couche est volontairement simpliste — sa limite (faible precision, aucun sens contextuel) est exactement ce qui motivera l'introduction du LLM dans les rungs suivants.

> **Important** : on travaille sur des **exemples synthetiques** (phrases abstraites construites pour la demonstration). Aucun texte reel, aucune donnee sensible.

In [5]:
# Cellule [6] - Detecteur deterministe par mots-cles + test sur un texte synthetique

# Dictionnaire {nom_du_sophisme: (famille, sous_famille, [mots_cles])}
# Les mots-cles sont choisis pour etre discriminants sur des exemples pedagogiques.
# Cette table est VOLONTAIREMENT courte : son but est pedagogique, pas exhaustif.
DETECTEUR_SOPHISMES = {
    "Generalisation abusive": ("Insuffisance", "Generalisation abusive",
                               ["tous les", "toujours", "jamais", "chaque", "aucun"]),
    "Ad hominem": ("Obstruction", "Ad hominem",
                   ["vous etes", "incompetent", "malhonnete", "qui etes-vous"]),
    "Appel a l autorite": ("Influence", "Appel a l autorite",
                           ["expert", "professeur", "autorite", "selon lui", "comme dit"]),
    "Faux dilemme": ("Erreur de raisonnement", "Faux dilemme",
                     ["soit ... soit", "ou bien", "seules options", "le seul choix"]),
}

def detecter_sophismes(texte, table=DETECTEUR_SOPHISMES):
    """Retourne la liste des sophismes detectes dans `texte` (matching insensible a la casse).

    Chaque element est un dict {sophisme, famille, sous_famille, mot_cle_trigger}.
    Aucun appel LLM : c'est du filtrage de chaine pur.
    """
    texte_lower = texte.lower()
    matches = []
    for nom_sophisme, (famille, sous_famille, mots_cles) in table.items():
        for mot in mots_cles:
            if mot.lower() in texte_lower:
                matches.append({
                    "sophisme": nom_sophisme,
                    "famille": famille,
                    "sous_famille": sous_famille,
                    "mot_cle_trigger": mot,
                })
                break  # un seul trigger par sophisme suffit
    return matches


# Texte synthetique neutre : pas de personne reelle, pas d opinion politique.
TEXTE_SYNTHETIQUE = (
    "Tous les politiciens mentent, donc celui-ci ment aussi. "
    "D apres le professeur X, c est evident. "
    "Vous etes incompetent pour dire le contraire."
)

detections = detecter_sophismes(TEXTE_SYNTHETIQUE)
print("--- Texte synthetique analyse ---")
print(TEXTE_SYNTHETIQUE)
print()
print(f"--- Sophismes detectes ({len(detections)}) ---")
for d in detections:
    print(f"  - {d['sophisme']:<25} (famille={d['famille']}, sous_famille={d['sous_famille']}) "
          f"trigger='{d['mot_cle_trigger']}'")

--- Texte synthetique analyse ---
Tous les politiciens mentent, donc celui-ci ment aussi. D apres le professeur X, c est evident. Vous etes incompetent pour dire le contraire.

--- Sophismes detectes (3) ---
  - Generalisation abusive    (famille=Insuffisance, sous_famille=Generalisation abusive) trigger='tous les'
  - Ad hominem                (famille=Obstruction, sous_famille=Ad hominem) trigger='vous etes'
  - Appel a l autorite        (famille=Influence, sous_famille=Appel a l autorite) trigger='professeur'


### Interpretation : sortie du detecteur

**Sortie obtenue** : 3 sophismes signales sur le texte synthetique, chacun avec son trigger.

| Sophisme | Famille | Trigger | Lecture |
|----------|---------|---------|--------|
| Generalisation abusive | Insuffisance | `tous les` | Le `tous les` indique une generalisation sans nuance |
| Appel a l'autorite | Influence | `professeur` | Invoquer une autorite ("professeur X") sans preuve |
| Ad hominem | Obstruction | `incompetent` | Attaque la personne, pas l'argument |

**Limites volontaires de ce detecteur** :
1. **Pas de contexte** : `tous les oiseaux ont des plumes` declencherait aussi "Generalisation abusive", alors que c'est un fait. Le matching mots-cles ignore la sémantique.
2. **Couverture limitee** : la table contient 4 sophismes ; la taxonomie en compte des centaines. Etendre la table a la main est fastidieux — c'est la qu'un LLM devient utile.
3. **Aucun score de confiance** : un match est binaire (present / absent). Pas de probabilite ni de ranking.

> **Lien avec les rungs suivants** : le rung 2 (logique formelle) verifie si un argument est valide ; le rung 3 (orchestration LLM) remplace la table mots-cles par une extraction semantique robuste. Ce rung montre la **ligne de base** (baseline) que ces techniques cherchent a ameliorer.

### Exercice 2 : ajouter une regle de detection pour un sophisme au choix

**Contexte** : la table `DETECTEUR_SOPHISMES` contient 4 entrees. Vous allez l'etendre.

**Objectif** : ajoutez une nouvelle entree dans le dictionnaire (un sophisme de votre choix parmi ceux de la taxonomie), definissez 2-3 mots-cles pertinents, puis testez votre detecteur etendu sur une phrase synthetique que vous ecrivez.

**Exemples de sophismes candidats** (a chercher dans le CSV filtre par `nom_vulgarisé`) : Pente glissante, Homme de paille, Appel a l'emotion, Argument circulaire.

**Indices** :
- # Etape 1 : copier `DETECTEUR_SOPHISMES` dans une nouvelle variable `table_etendue`
- # Etape 2 : ajouter une cle avec un tuple `(famille, sous_famille, [mots_cles])`
- # Etape 3 : ecrire une `PHRASE_TEST` synthetique contenant un des mots-cles
- # Etape 4 : appeler `detecter_sophismes(PHRASE_TEST, table_etendue)` et afficher

In [6]:
# Exercice 2 : etendre le detecteur et tester sur une phrase synthetique.
# Etape 1 : copier la table existante
table_etendue = dict(DETECTEUR_SOPHISMES)  # TODO etudiant : point de depart

# Etape 2 : ajouter un nouveau sophisme
# TODO etudiant : table_etendue['Mon sophisme'] = ('Famille', 'Sous-famille', ['mot1', 'mot2'])

# Etape 3 : phrase synthetique de test
PHRASE_TEST = None  # TODO etudiant : une phrase contenant un des nouveaux mots-cles

# Etape 4 : detection et affichage
detections_ex2 = []  # TODO etudiant : detecter_sophismes(PHRASE_TEST, table_etendue)
print(f"Exercice 2 : {len(detections_ex2)} sophisme(s) detecte(s)")
print("Exercice a completer")

Exercice 2 : 0 sophisme(s) detecte(s)
Exercice a completer


### Exercice 3 : fonction qui retourne la famille du premier sophisme detecte

**Contexte** : on veut parfois une reponse simple — "quelle est la famille dominante de ce texte ?" — plutot qu'une liste de tous les matches.

**Objectif** : ecrivez une fonction `famille_premier_sophisme(texte)` qui retourne le nom de la **famille** du premier sophisme detecte dans `texte`, ou `None` si aucun match. Utilisez `detecter_sophismes` definie plus haut.

**Indices** :
- # Etape 1 : appeler `detecter_sophismes(texte)` pour obtenir la liste des matches
- # Etape 2 : si la liste est vide, retourner `None`
- # Etape 3 : sinon, retourner `matches[0]['famille']`
- # Etape 4 : tester sur deux phrases (une qui match, une qui ne match pas)

In [7]:
# Exercice 3 : famille du premier sophisme detecte.
def famille_premier_sophisme(texte):
    # TODO etudiant : utiliser detecter_sophismes(texte) et retourner
    # la famille du premier element, ou None si la liste est vide.
    # Etape 1 :
    matches = []  # TODO etudiant : detecter_sophismes(texte)
    # Etape 2-3 :
    return None  # TODO etudiant : remplacer par la logique

# Etape 4 : tests
phrase_avec = "Tous les chats sont noirs, c'est bien connu."
phrase_sans = "Le ciel est bleu aujourd'hui."
famille_avec = famille_premier_sophisme(phrase_avec)
famille_sans = famille_premier_sophisme(phrase_sans)
print(f"Phrase avec sophisme -> famille : {famille_avec}")
print(f"Phrase sans sophisme  -> famille : {famille_sans}")
print("Exercice a completer")

Phrase avec sophisme -> famille : None
Phrase sans sophisme  -> famille : None
Exercice a completer


## 5. Pont vers l'etat partage (shim argumentation_lib)

Jusqu'ici, les detections de la section 4 vivent dans une **liste Python locale**. Dans le pipeline complet de la serie, les analyses (informelle, formelle, orchestree) partagent un **etat unique** -- l'objet `RhetoricalAnalysisState` fourni par la shim `argumentation_lib`. C'est l'objet que le rung [3-orchestration](./Argument_Analysis_Agentic-3-orchestration.ipynb) remplit via un agent LLM, et que le rung [2-formal](./Argument_Analysis_Agentic-2-formal.ipynb) enrichit de resultats de preuve.

Cette section montre le **geste fondamental** du pipeline : nourrir cet etat partage a partir de notre detecteur deterministe -- **sans LLM**. On retrouve ainsi, cote a cote, les memes structures de donnees que les rungs ulterieurs manipuleront.

| Champ de l'etat | Rempli ici (detecteur mots-cles) | Rempli au rung 3 (LLM) |
|-----------------|----------------------------------|------------------------|
| `raw_text` | le texte synthetique | un texte reel fourni en entree |
| `identified_arguments` | un argument de demonstration | les arguments extraits semantiquement |
| `identified_fallacies` | les detections + leur famille | les sophismes extraits par le LLM |

> **Pourquoi anticiper cette structure ?** La valeur d'un pipeline multi-agents tient dans l'**etat partage** : chaque agent (detection, formalisation, synthese) lit et enrichit le meme objet. Brancher notre baseline dessus rend cette architecture concrete bien avant d'introduire le LLM.

In [8]:
# Cellule [section 5] - Pont vers l'etat partage : nourrir RhetoricalAnalysisState
#
# Le detecteur mots-cles (section 4) produit une liste Python locale. Dans le
# pipeline complet de la serie, les analyses (informelle, formelle, orchestree)
# partagent un ETAT UNIQUE -- l'objet RhetoricalAnalysisState de la shim
# argumentation_lib. Le rung 3 le remplit via un agent LLM ; le rung 2 y attache
# des resultats de preuve. Ici on montre le geste fondamental : nourrir cet
# etat a partir de notre detecteur deterministe, SANS LLM.
import logging
from argumentation_lib import RhetoricalAnalysisState

# Rendre les logs internes de l'etat silencieux (niveau WARNING) pour un output propre.
logging.getLogger("RhetoricalAnalysisState").setLevel(logging.WARNING)

etat = RhetoricalAnalysisState(initial_text=TEXTE_SYNTHETIQUE)
arg_id = etat.add_argument("Argument synthetique analyse par le detecteur mots-cles")
for d in detections:
    etat.add_fallacy(
        fallacy_type=d["sophisme"],
        justification=f"declencheur mot-cle : '{d['mot_cle_trigger']}'",
        target_arg_id=arg_id,
        family=d["famille"],
        taxonomy_path=d["sous_famille"],
    )

snapshot = etat.get_state_snapshot(summarize=True)
print("--- Etat partage apres detection mots-cles ---")
print(f"  raw_text             : {snapshot['raw_text_snippet']}")
print(f"  arguments identifies : {snapshot['argument_count']}  -> {list(etat.identified_arguments)}")
print(f"  sophismes identifies : {snapshot['fallacy_count']}")
for fid, fdata in etat.identified_fallacies.items():
    print(f"     {fid} : {fdata['type']:<25} (famille={fdata.get('family','?')}, cible={fdata.get('target_argument_id','?')})")
print()
print("Cet etat est exactement la structure que le rung 3 (orchestration LLM)")
print("remplira via un agent, et que le rung 2 (formel) enrichira de resultats de preuve.")

--- Etat partage apres detection mots-cles ---
  raw_text             : Tous les politiciens mentent, donc celui-ci ment aussi. D apres le professeur X, c est evident. Vous etes incompetent pour dire le contraire.
  arguments identifies : 1  -> ['arg_1']
  sophismes identifies : 3
     fallacy_1 : Generalisation abusive    (famille=Insuffisance, cible=arg_1)
     fallacy_2 : Ad hominem                (famille=Obstruction, cible=arg_1)
     fallacy_3 : Appel a l autorite        (famille=Influence, cible=arg_1)

Cet etat est exactement la structure que le rung 3 (orchestration LLM)
remplira via un agent, et que le rung 2 (formel) enrichira de resultats de preuve.


## 6. Conclusion et recapitulatif

Ce rung a pose les **fondations taxonomiques** de la detection de sophismes : on charge un referentiel structure (le CSV), on descend dans l'arbre par filtre pandas, puis on construit une couche de matching mots-cles. La section 5 a de plus **branche cette baseline sur l'etat partage** de la serie (`RhetoricalAnalysisState`). Aucune LLM, aucune simulation : tout est deterministe et reproductible.

### Tableau recapitulatif

| Etape | Outil | Force | Limite |
|-------|-------|-------|-------|
| **Charger la taxonomie** | `pandas.read_csv` + `argumentation_lib.get_data_dir()` | Referentiel de 1408 sophismes + 222 vertus verifiees, path cwd-robuste | Le CSV est vaste (1408 lignes), il faut filtrer |
| **Charger la taxonomie soeur Vertues** | Meme shim, dispatch filename-based | Schema distinct delibere (snake_case, 3-field body, 222 cartes) | Pas d'unified schema inter-CSV -- KeyError si presume |
| **Descendre d'un niveau** | Filtre booleen `(Famille == X) & (depth == '2')` | Identification des sous-familles directes | Manuelle, famille par famille |
| **Detection mots-cles** | Dictionnaire `{sophisme: [mots]}` + `in` | Deterministe, zero API, transparent | Pas de contexte, faible precision |
| **Etats partage** | `RhetoricalAnalysisState.add_fallacy(...)` | Branche la baseline sur le pipeline multi-agents | La structure reste vide tant qu'aucun agent LLM ne l'enrichit |

### Points a retenir

1. **La taxonomie est un arbre** : `depth` encode le niveau (0=racine meta, 1=familles, 2=sous-familles, 3+=instances specifiques). Le `path` (ex `7.3`) encode la position hierarchique.

2. **Le detecteur mots-cles est une baseline** : il montre ce qu'on peut faire sans LLM, et donc ce que le LLM devra ameliorer. Une limite importante est l'absence de contexte (`tous les oiseaux` declencherait une fausse "Generalisation abusive").

3. **Confidentialite par design** : tous les exemples sont synthetiques. Pour un usage reel, le pipeline complet (rung 3) orchestre l'extraction LLM puis la verification formelle (rung 2) -- jamais de stockage de texte sensible dans le depot public.

4. **L'etat partage est le lien entre les rungs** : `RhetoricalAnalysisState` est le conteneur commun. Ce rung (detection), le rung 2 (preuve) et le rung 3 (orchestration LLM) y lisent et ecrivent. Brancher notre baseline des maintenant rend cette architecture concrete.

### Prochaines etapes

- [2-formal](./Argument_Analysis_Agentic-2-formal.ipynb) : une fois un argument identifie, verifier sa **validite logique** avec le raisonneur Tweety (PL, FOL, modal, argumentation de Dung).
- [3-orchestration](./Argument_Analysis_Agentic-3-orchestration.ipynb) : remplacer la table mots-cles par un LLM qui extrait la structure informative, puis deleguer la verification a Tweety.